In [ ]:
#Sentence classification

import numpy as np


onehots = {
    'cat': np.array([1, 0, 0, 0]),
    'the': np.array([0, 1, 0, 0]),
    'dog': np.array([0, 0, 1, 0]),
    'sat': np.array([0, 0, 0, 1])
}


sentence = ['the', 'cat', 'sat']


sentence_encoding = np.zeros(4)

for word in sentence:
    sentence_encoding += onehots[word]


print("Sentence Encoding:", sentence_encoding)


Sentence Encoding: [1. 1. 0. 1.]


In [ ]:
f = open('reviews.txt')
raw_reviews = f.readlines()
f.close()
f = open('labels.txt')
raw_labels = f.readlines()
f.close()
tokens = []
for review in raw_reviews:
    words = review.split(" ")
    tokens.append(set(words))
print(len(tokens))

vocab = set()
for sent in tokens:
  for word in sent:
    if (len(word) > 0):
      vocab.add(word)
vocab = list(vocab)
print(vocab)

word2index={}
for i,word in enumerate(vocab):
  word2index[word]=i

print(word2index)

6
['acting', 'great', 'worst', 'absolutely', 'it', 'the', 'was', 'loved', 'inspiring\n', 'what', 'experience\n', 'terrible\n', 'film', 'plot', 'i', 'a', 'ever\n', 'fantastic', 'wonderful', 'story\n', 'and', 'hated', 'predictable\n', 'movie', 'boring', 'this']
{'acting': 0, 'great': 1, 'worst': 2, 'absolutely': 3, 'it': 4, 'the': 5, 'was': 6, 'loved': 7, 'inspiring\n': 8, 'what': 9, 'experience\n': 10, 'terrible\n': 11, 'film': 12, 'plot': 13, 'i': 14, 'a': 15, 'ever\n': 16, 'fantastic': 17, 'wonderful': 18, 'story\n': 19, 'and': 20, 'hated': 21, 'predictable\n': 22, 'movie': 23, 'boring': 24, 'this': 25}


In [ ]:
input_dataset = list()
for sent in tokens:
    sent_indices = list()
    for word in sent:
        try:
            sent_indices.append(word2index[word])
        except:
            pass
    input_dataset.append(list(set(sent_indices)))
print(input_dataset)

[[6, 8, 17, 20, 23, 25], [4, 6, 11, 12, 14, 21, 25], [0, 3, 5, 7, 19, 20], [5, 6, 13, 20, 22, 24], [1, 9, 10, 15, 18, 20], [2, 5, 6, 16, 23, 25]]


In [ ]:
target_dataset= list()
for lable in raw_labels:
  if lable == 'positive\n':
    target_dataset.append(1)
  else:
    target_dataset.append(0)


print(target_dataset)

[1, 0, 1, 0, 1, 0]


In [ ]:
import numpy as np
seed=42
def sigmoid(x):
  return 1/(1+np.exp(-x))

alpha=0.01
iterations=10
hidden_size=100

weights_0_1 = np.random.random((len(vocab), hidden_size))
weights_1_2 = np.random.random((hidden_size, 1))

for iter in range(iterations):
  for i in range(len(input_dataset)):
    x, y = (input_dataset[i], target_dataset[i])

    #forward propagation
    layer_1 = sigmoid(np.sum(weights_0_1[x], axis=0))
    layer_2 = sigmoid(np.dot(layer_1, weights_1_2))

    #backward propagation
    layer_2_delta = layer_2 - y
    layer_1_delta = layer_2_delta.dot(weights_1_2.T)

    #update weights
    weights_0_1[x] -= layer_1_delta * alpha
    layer_1 = layer_1.reshape(hidden_size, 1)
    layer_2_delta = layer_2_delta.reshape(1, 1)
    weights_1_2 -= np.dot(layer_1, layer_2_delta) * alpha

correct, total = (0, 0)
for i in range(len(input_dataset)):
  x = input_dataset[i]
  y = target_dataset[i]
  layer_1 = sigmoid(np.sum(weights_0_1[x], axis=0))
  layer_2 = sigmoid(np.dot(layer_1, weights_1_2))
  if (np.abs(layer_2 - y) < 0.5):
    correct += 1
  total += 1

output=correct/(total)
print(output)

0.5


In [ ]:
from collections import Counter
import math
def similar(target="fantastic"):
  target_index = word2index[target]
  scores = Counter()
  for word, index in word2index.items():
    raw_difference = weights_0_1[index] - (weights_0_1[target_index])
    squared_difference = raw_difference * raw_difference
    scores[word] = -math.sqrt(sum(squared_difference))
  return scores.most_common(10)
print(similar("fantastic"))
print(similar("boring"))

[('fantastic', -0.0), ('this', -3.750107165918774), ('film', -3.8082531829044064), ('a', -3.849767091211709), ('boring', -3.9614359522350506), ('absolutely', -3.9792678671804813), ('and', -3.9873555340654083), ('experience\n', -4.060278237190131), ('inspiring\n', -4.076136219044794), ('terrible\n', -4.077193946804266)]
[('boring', -0.0), ('absolutely', -3.435545785463909), ('it', -3.5630259189762223), ('what', -3.609001613165123), ('story\n', -3.8112345277724344), ('ever\n', -3.8832078075615564), ('acting', -3.884077577600045), ('this', -3.9422686258471376), ('predictable\n', -3.944269589485104), ('fantastic', -3.9614359522350506)]


In [ ]:
#Word2vec

import random as random
np.random.seed(42)
random.seed(42)

concatenated = []
input_dataset = []
for sent in tokens:
  sent_indices = []
  for word in sent:
    try:
      sent_indices.append(word2index[word])
      concatenated.append(word2index[word])
    except:
      ""
  input_dataset.append(sent_indices)
concatenated = np.array(concatenated)
random.shuffle(input_dataset)


In [ ]:
alpha, iterations = (0.05, 100)
hidden_size, window, negative = (50, 2, 5)
weights_0_1 = np.random.rand(len(vocab), hidden_size)
weights_1_2 = np.random.rand(len(vocab), hidden_size)
layer_2_target = np.zeros(negative + 1)
layer_2_target[0] = 1
def sigmoid(x):
  return 1 / (1 + np.exp(-x))

In [ ]:
for rev_i, review in enumerate(input_dataset * iterations):
  for target_i in range(len(review)):
    target_samples = [review[target_i]] + list(concatenated[(np.random.rand(negative) * len(concatenated)) .astype('int')].tolist())
    left_context = review[max(0, target_i - window):target_i]
    right_context = review[target_i + 1:min(len(review), target_i + window)]
    layer_1 = np.mean(weights_0_1[left_context + right_context], axis=0)
    layer_2 = sigmoid(layer_1.dot(weights_1_2[target_samples].T))
    layer_2_delta = layer_2 - layer_2_target
    layer_1_delta = layer_2_delta.dot(weights_1_2[target_samples])
    weights_0_1[left_context + right_context] -= layer_1_delta * alpha
    layer_2_delta = layer_2_delta.reshape(-1, 1)
    layer_1 = layer_1.reshape(1, -1)
    weights_1_2[target_samples] -= np.dot(layer_2_delta, layer_1) * alpha

def analogy(positive, negative):
  norms = np.sum(weights_0_1 * weights_0_1, axis=1)
  norms.resize(norms.shape[0], 1)
  normed_weights = weights_0_1 * norms
  query_vec = np.zeros(len(weights_0_1[0]))
  for word in positive:
    query_vec += normed_weights[word2index[word]]
  for word in negative:
    query_vec -= normed_weights[word2index[word]]
  scores = Counter()
  for word, index in word2index.items():
    raw_difference = weights_0_1[index] - query_vec
    squared_difference = raw_difference * raw_difference
    scores[word] = -math.sqrt(sum(squared_difference))
  return scores.most_common(10)[1:]

print(analogy(['great', 'wonderful'],['fantastic']))

[('was', -688.0782929752152), ('absolutely', -688.5350992982529), ('terrible\n', -688.9881332837164), ('this', -689.2154137380493), ('a', -689.2268995196138), ('story\n', -689.2536909144128), ('the', -689.5184884401503), ('boring', -689.8934902897147), ('it', -690.3847779497362)]
